# v2.5 — High-recall detection + physical NMS (local eval)

Borrows the LB 0.843 reference's recipe on the **detection side**, keeping the v2 rule-based linking:
- **Detect for max recall**: lower `HM_THR` (0.3 -> 0.15) and `min_distance=1` (let physical NMS dedup
  instead of an anisotropic voxel gate).
- **Physical NMS** (`cKDTree`, micrometre space): greedy keep-highest-peak, drop only neighbours within
  `NMS_UM` -> removes *obvious* duplicates without the anisotropy of voxel `min_distance`.

**UPDATE (post-LB): linking REVERTED to v2, and it is the single variable under test.** The first v2.5
submission (relaxed linking: loose 10 um, `filter_short_tracks` min_len 2) scored **LB 0.776 — WORSE than
v2's 0.827** even though local edge-J rose 0.859 -> 0.866. The local metric is blind to (a) FP edges on the
dense hidden GT and (b) the leaderboard's *adjusted* density penalty, so the relaxed linking looked free
locally but cost real precision on the LB. This notebook now **freezes v2.5 high-recall detection + NMS**
and **restores v2's LB-proven linking precision** (`LOOSE_UM=8`, `FILTER_MIN_LEN=4`) — the ONLY change vs
the 0.776 run. Diagnostic: if LB recovers to ~0.827+, the relaxed linking was the culprit and high-recall
detection is fine; if it stays ~0.776, the detection density is the culprit -> lock in v2.

Everything else (v1 UNet weights, two-pass motion Hungarian, close_gaps) is the v2 stack.

**Baseline to beat:** v2 = **0.859** local (LB 0.827), same detector, same val samples.
Watch `matched/n_gt` (detection recall) and `pred_nodes`, and especially **FP** — the whole point of the
revert is to push FP back toward v2's ~0.

**Note on runtime:** low threshold + `min_distance=1` produces many more peaks (the 0.843 ref hit
~2k-8k nodes/frame), so Hungarian linking is slower. This is a local eval on a few samples; fine to
wait. Dial `HM_THR` back up if it is too slow.

**Setup:** add competition data + `v1_UNet_best.pt` as inputs. GPU ON, Internet ON (installs zarr). Run All.

In [ ]:
# --- deps ---
import subprocess, sys
try:
    import zarr; assert int(zarr.__version__.split('.')[0]) >= 3
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'zarr>=3']); import zarr

import os, glob, time
from collections import defaultdict
from dataclasses import dataclass
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from skimage.feature import peak_local_max
print('zarr', zarr.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# ================= CONFIG =================
INPUT     = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TRAIN_DIR = os.path.join(INPUT, 'train')
VOXEL = np.array([1.625, 0.40625, 0.40625])   # (z,y,x) um/voxel (anisotropic)
SCALE = VOXEL
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- UNet architecture: MUST match v1_UNet_best.pt ---
BASE    = 16
STRIDES = ((1,2,2), (1,2,2), (2,2,2), (2,2,2))

# --- detection: HIGH-RECALL preset (FROZEN = v2.5; single-variable experiment) ---
NORM_PCT   = (50.0, 99.8)
HM_THR     = 0.15         # v2.5, unchanged: lowered from 0.3 -> more peaks -> higher recall
MIN_DIST   = 1            # v2.5, unchanged: 3 -> 1; rely on physical NMS below, not the anisotropic voxel gate
MAX_PEAKS  = 200000       # per-frame cap (won't usually bind)
NMS_UM     = 3.0          # v2.5, unchanged: physical (um) NMS radius; only removes obvious duplicates. 0 = off
REFINE     = True
REFINE_WIN = (1, 3, 3)

# --- linking + post-processing: REVERTED to v2 (the ONE variable under test) ---
# v2.5 relaxed linking (loose 10 / min_len 2) coincided with the LB DROP 0.827 -> 0.776 while local rose
# (local eval is blind to FP edges on the dense hidden GT + the adjusted density penalty). This run keeps
# v2.5 high-recall detection but restores v2's LB-proven linking precision to isolate the cause.
TIGHT_UM   = 6.0          # tight gate (unchanged)
LOOSE_UM   = 8.0          # REVERT v2.5 10 -> 8 (v2 value; tighten linking precision)
VEL_BLEND  = 0.5
CLOSE_GAPS = True
MAX_GAP    = 1
GAP_DIST_UM= 6.0
FILTER_MIN_LEN = 4        # REVERT v2.5 2 -> 4 (v2 value; drop spurious short tracks = FP edges)
PRUNE_ISOLATED = True

# --- eval ---
N_VAL        = 20
EVAL_N_VAL   = 5          # raise toward 20 for a firmer, specimen-balanced number
MATCH_UM     = 7.0
EVAL_WEIGHTS = None

In [ ]:
# ================= IO (zarr, same as v1/v2) =================
def open_image(zarr_path):
    g = zarr.open(zarr_path, mode='r')
    if hasattr(g, 'shape'):
        return g
    best, bestsize = None, -1
    def walk(node):
        nonlocal best, bestsize
        try:
            for k in node.keys():
                sub = node[k]
                if hasattr(sub, 'shape'):
                    s = int(np.prod(sub.shape))
                    if s > bestsize: best, bestsize = sub, s
                else: walk(sub)
        except Exception: pass
    walk(g); return best

def load_geff(geff_path):
    g = zarr.open(geff_path, mode='r')
    node_ids = np.asarray(g['nodes/ids'][:])
    props = {k: np.asarray(g[f'nodes/props/{k}/values'][:]) for k in g['nodes/props'].keys()}
    try: edges = np.asarray(g['edges/ids'][:])
    except Exception: edges = np.zeros((0, 2), dtype=np.uint64)
    return node_ids, props, edges

def gt_nodes_edges(geff_path):
    node_ids, props, edges = load_geff(geff_path)
    nodes = [(int(node_ids[i]), int(props['t'][i]), int(props['z'][i]), int(props['y'][i]), int(props['x'][i]))
             for i in range(len(node_ids))]
    return nodes, [(int(s), int(t)) for s, t in edges]

def build_sample_list(train_dir):
    out = []
    for gp in sorted(glob.glob(os.path.join(train_dir, '*.geff'))):
        zp = gp[:-5] + '.zarr'
        if os.path.exists(zp): out.append((zp, gp))
    return out

In [ ]:
# ================= MODEL: anisotropic 3D U-Net (verbatim from v1) =================
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(cin, cout, 3, padding=1, bias=False), nn.InstanceNorm3d(cout, affine=True), nn.LeakyReLU(0.01, True),
            nn.Conv3d(cout, cout, 3, padding=1, bias=False), nn.InstanceNorm3d(cout, affine=True), nn.LeakyReLU(0.01, True))
    def forward(self, x): return self.net(x)

class UNet3D(nn.Module):
    def __init__(self, in_ch=1, base=BASE, strides=STRIDES):
        super().__init__()
        n = len(strides); chs = [base*(2**i) for i in range(n+1)]
        self.enc, self.downs = nn.ModuleList(), nn.ModuleList()
        cin = in_ch
        for i in range(n):
            self.enc.append(ConvBlock(cin, chs[i]))
            self.downs.append(nn.Conv3d(chs[i], chs[i], kernel_size=strides[i], stride=strides[i]))
            cin = chs[i]
        self.bottleneck = ConvBlock(chs[n-1], chs[n])
        self.ups, self.dec = nn.ModuleList(), nn.ModuleList()
        for i in reversed(range(n)):
            self.ups.append(nn.ConvTranspose3d(chs[i+1], chs[i], kernel_size=strides[i], stride=strides[i]))
            self.dec.append(ConvBlock(chs[i]*2, chs[i]))
        self.head = nn.Conv3d(chs[0], 1, 1)
    def forward(self, x):
        skips = []
        for enc, down in zip(self.enc, self.downs):
            x = enc(x); skips.append(x); x = down(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.ups, self.dec, reversed(skips)):
            x = up(x)
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        return self.head(x)

def find_weights():
    if EVAL_WEIGHTS: return EVAL_WEIGHTS
    for pat in ('v1_UNet_best.pt', 'unet_heatmap.pt', 'unet_latest.pt', '*.pt'):
        hits = sorted(glob.glob(f'/kaggle/input/**/{pat}', recursive=True))
        if hits: return hits[0]
    return None

def extract_state(ck):
    if isinstance(ck, dict) and ('best_model' in ck or 'model' in ck):
        return ck.get('best_model') or ck['model']
    return ck

In [ ]:
# ================= DETECTION: full-res UNet heatmap -> peaks -> physical NMS =================
def _normalize(vol, norm_pct=NORM_PCT):
    v = vol.astype(np.float32)
    lo, hi = np.percentile(v, norm_pct)
    return np.clip((v - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)

@torch.no_grad()
def predict_heatmap(model, vol):
    v = _normalize(vol)
    x = torch.from_numpy(np.ascontiguousarray(v)[None, None]).float().to(device)
    with torch.amp.autocast(device, enabled=(device == 'cuda')):
        y = torch.sigmoid(model(x))
    return y[0, 0].float().cpu().numpy()

def physical_nms(coords, scores, radius_um, scale=VOXEL):
    """Greedy non-max suppression in PHYSICAL (um) space: keep the strongest peak, drop others
    within radius_um. Isotropic in real units, unlike a voxel-space min_distance."""
    if len(coords) <= 1 or not radius_um:
        return coords, scores
    pts = coords * scale[None, :]
    order = np.argsort(-scores)
    tree = cKDTree(pts)
    killed = np.zeros(len(coords), bool); keep = []
    for i in order:
        if killed[i]: continue
        keep.append(int(i))
        killed[tree.query_ball_point(pts[i], r=radius_um)] = True   # suppresses i itself + neighbours
    keep = np.array(keep)
    return coords[keep], scores[keep]

def detect_frames(model, arr, thr=HM_THR, min_distance=MIN_DIST, max_peaks=MAX_PEAKS,
                  nms_um=NMS_UM, refine=REFINE, win=REFINE_WIN):
    """Per-frame: heatmap -> peak_local_max -> (refine) -> physical NMS. Returns list of (N,3) coords."""
    model.eval(); frames = []
    for t in range(arr.shape[0]):
        vol = np.asarray(arr[t]).astype(np.float32)
        hm = predict_heatmap(model, vol)
        pk = peak_local_max(hm, min_distance=min_distance, threshold_abs=thr,
                            exclude_border=False, num_peaks=max_peaks)
        if len(pk) == 0:
            frames.append(np.zeros((0, 3))); continue
        scores = hm[pk[:, 0], pk[:, 1], pk[:, 2]].astype(float)
        coords = pk.astype(np.float64)
        if refine:
            coords = refine_centroids(vol, coords, win=win)
        coords, scores = physical_nms(coords, scores, nms_um)
        frames.append(coords)
    return frames

In [ ]:
# ================= LINKING + POST-PROCESSING (rule-based v2 stack) =================
@dataclass
class TrackGraph:
    node_t: np.ndarray; node_z: np.ndarray; node_y: np.ndarray; node_x: np.ndarray
    node_ids: np.ndarray; edges: np.ndarray; meta: dict
    @property
    def n_nodes(self): return len(self.node_ids)
    @property
    def n_edges(self): return len(self.edges)

def refine_centroids(vol, coords, win=(1, 3, 3)):
    if len(coords) == 0:
        return coords
    Z, Y, X = vol.shape
    out = coords.copy().astype(np.float64)
    wz, wy, wx = win
    for i, (z, y, x) in enumerate(coords):
        z, y, x = int(round(z)), int(round(y)), int(round(x))
        z0, z1 = max(0, z - wz), min(Z, z + wz + 1)
        y0, y1 = max(0, y - wy), min(Y, y + wy + 1)
        x0, x1 = max(0, x - wx), min(X, x + wx + 1)
        patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float64)
        s = patch.sum()
        if s <= 0: continue
        zz = np.arange(z0, z1)[:, None, None]
        yy = np.arange(y0, y1)[None, :, None]
        xx = np.arange(x0, x1)[None, None, :]
        out[i, 0] = (patch * zz).sum() / s
        out[i, 1] = (patch * yy).sum() / s
        out[i, 2] = (patch * xx).sum() / s
    return out

def link_twopass(frames, tight_um=6.0, loose_um=8.0, vel_blend=0.5):
    node_ids = []; node_t = []; node_z = []; node_y = []; node_x = []; frame_ids = []; nid = 1
    for t, coords in enumerate(frames):
        ids = []
        for (z, y, x) in coords:
            node_ids.append(nid); node_t.append(t); node_z.append(z); node_y.append(y); node_x.append(x)
            ids.append(nid); nid += 1
        frame_ids.append(ids)
    def _hun(P, C, pred, pi, ci, gate):
        if len(pi) == 0 or len(ci) == 0: return []
        BIG = 1e9
        Draw = np.sqrt(((P[pi][:, None] - C[ci][None]) ** 2).sum(2))
        D = np.sqrt(((pred[pi][:, None] - C[ci][None]) ** 2).sum(2))
        cost = np.where(Draw > gate, BIG, D)
        ri, rc = linear_sum_assignment(cost)
        return [(int(pi[r]), int(ci[c])) for r, c in zip(ri, rc) if cost[r, c] < BIG]
    edges = []; prev_xyz = np.zeros((0, 3)); prev_ids = []; prev_vel = None
    for t in range(len(frames)):
        curr = np.asarray(frames[t], float).reshape(-1, 3); curr_ids = frame_ids[t]
        if t > 0 and len(prev_ids) and len(curr):
            P = prev_xyz * SCALE[None, :]; C = curr * SCALE[None, :]
            pred = P + (vel_blend * prev_vel if prev_vel is not None else 0.0)
            N, M = len(P), len(C)
            links = _hun(P, C, pred, np.arange(N), np.arange(M), min(tight_um, loose_um))
            up = {p for p, _ in links}; uc = {c for _, c in links}
            fp = np.array([i for i in range(N) if i not in up], int)
            fc = np.array([j for j in range(M) if j not in uc], int)
            links += _hun(P, C, pred, fp, fc, loose_um)
            vel = np.zeros((N, 3)); nv = np.zeros((M, 3))
            for p, c in links:
                edges.append((prev_ids[p], curr_ids[c])); vel[p] = (curr[c] - prev_xyz[p]) * SCALE
            for p, c in links:
                nv[c] = vel[p]
            prev_vel = nv
        else:
            prev_vel = None
        prev_xyz, prev_ids = curr, curr_ids
    return TrackGraph(node_t=np.array(node_t, np.int64), node_z=np.array(node_z, float),
                      node_y=np.array(node_y, float), node_x=np.array(node_x, float),
                      node_ids=np.array(node_ids, np.int64),
                      edges=np.array(edges, np.int64).reshape(-1, 2), meta={})

def close_gaps(frames, g, max_gap=1, gap_dist_um=8.0):
    if g.n_edges == 0:
        return g
    coords = {int(nid): (int(g.node_t[i]), g.node_z[i], g.node_y[i], g.node_x[i])
              for i, nid in enumerate(g.node_ids)}
    has_out = set(int(s) for s, _ in g.edges)
    has_in = set(int(t) for _, t in g.edges)
    ends_by_t = {}; starts_by_t = {}
    for nid, (t, z, y, x) in coords.items():
        if nid not in has_out: ends_by_t.setdefault(t, []).append(nid)
        if nid not in has_in: starts_by_t.setdefault(t, []).append(nid)
    new_nodes = []; new_edges = []
    next_id = int(g.node_ids.max()) + 1 if g.n_nodes else 1
    for gap in range(1, max_gap + 1):
        for t, ends in ends_by_t.items():
            starts = starts_by_t.get(t + gap + 1)
            if not starts: continue
            ec = np.array([[coords[e][1], coords[e][2], coords[e][3]] for e in ends]) * SCALE
            sc = np.array([[coords[s][1], coords[s][2], coords[s][3]] for s in starts]) * SCALE
            d = np.sqrt(((ec[:, None, :] - sc[None, :, :]) ** 2).sum(axis=2))
            thr = gap_dist_um * (gap + 1)
            big = thr * 1000 + 1
            cost = np.where(d <= thr, d, big)
            ri, ci = linear_sum_assignment(cost)
            used_s = set()
            for r, c in zip(ri, ci):
                if d[r, c] > thr or ends[r] in has_out or starts[c] in used_s: continue
                e_id, s_id = ends[r], starts[c]
                te, ze, ye, xe = coords[e_id]; ts, zs, ys, xs = coords[s_id]
                prev = e_id
                for k in range(1, gap + 1):
                    frac = k / (gap + 1)
                    zi = ze + (zs - ze) * frac; yi = ye + (ys - ye) * frac; xi = xe + (xs - xe) * frac
                    nid = next_id; next_id += 1
                    new_nodes.append((te + k, zi, yi, xi, nid)); new_edges.append((prev, nid)); prev = nid
                new_edges.append((prev, s_id)); has_out.add(e_id); used_s.add(c)
    if not new_nodes:
        return g
    nt = np.concatenate([g.node_t, np.array([n[0] for n in new_nodes], dtype=np.int64)])
    nz = np.concatenate([g.node_z, np.array([n[1] for n in new_nodes])])
    ny = np.concatenate([g.node_y, np.array([n[2] for n in new_nodes])])
    nx = np.concatenate([g.node_x, np.array([n[3] for n in new_nodes])])
    nid = np.concatenate([g.node_ids, np.array([n[4] for n in new_nodes], dtype=np.int64)])
    edges = np.concatenate([g.edges, np.array(new_edges, dtype=np.int64).reshape(-1, 2)])
    return TrackGraph(node_t=nt, node_z=nz, node_y=ny, node_x=nx, node_ids=nid, edges=edges, meta=g.meta)

def prune_isolated(g):
    if g.n_edges == 0: return g
    used = set(int(x) for x in g.edges.reshape(-1))
    keep = np.array([i for i, nid in enumerate(g.node_ids) if int(nid) in used])
    if len(keep) == len(g.node_ids): return g
    return TrackGraph(node_t=g.node_t[keep], node_z=g.node_z[keep], node_y=g.node_y[keep],
                      node_x=g.node_x[keep], node_ids=g.node_ids[keep], edges=g.edges, meta=g.meta)

def filter_short_tracks(g, min_len):
    if g.n_edges == 0 or min_len <= 1: return g
    parent = {int(n): int(n) for n in g.node_ids}
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for s, t in g.edges.reshape(-1, 2):
        ra, rb = find(int(s)), find(int(t))
        if ra != rb: parent[ra] = rb
    comp = {}
    for n in g.node_ids:
        comp.setdefault(find(int(n)), []).append(int(n))
    keep = set()
    for members in comp.values():
        if len(members) >= min_len: keep.update(members)
    idx = [i for i, n in enumerate(g.node_ids) if int(n) in keep]
    keepset = set(int(g.node_ids[i]) for i in idx)
    edges = np.array([(int(s), int(t)) for s, t in g.edges.reshape(-1, 2)
                      if int(s) in keepset and int(t) in keepset], dtype=np.int64).reshape(-1, 2)
    return TrackGraph(node_t=g.node_t[idx], node_z=g.node_z[idx], node_y=g.node_y[idx],
                      node_x=g.node_x[idx], node_ids=g.node_ids[idx], edges=edges, meta=g.meta)

def graph_to_nodes_edges(g):
    nodes = [(int(g.node_ids[i]), int(g.node_t[i]), int(round(g.node_z[i])),
              int(round(g.node_y[i])), int(round(g.node_x[i]))) for i in range(g.n_nodes)]
    edges = [(int(s), int(t)) for s, t in g.edges]
    return nodes, edges

In [ ]:
# ================= METRIC: official edge-Jaccard (copied from v1) =================
def match_nodes_per_t(gt_nodes, pred_nodes, max_um=MATCH_UM):
    gt_by_t, pr_by_t = defaultdict(list), defaultdict(list)
    for n in gt_nodes: gt_by_t[n[1]].append(n)
    for n in pred_nodes: pr_by_t[n[1]].append(n)
    g2p = {}
    for t, G in gt_by_t.items():
        P = pr_by_t.get(t, [])
        if not P: continue
        A = np.array([[g[2], g[3], g[4]] for g in G]) * VOXEL
        B = np.array([[p[2], p[3], p[4]] for p in P]) * VOXEL
        D = cdist(A, B); r, c = linear_sum_assignment(D)
        for i, j in zip(r, c):
            if D[i, j] <= max_um: g2p[G[i][0]] = P[j][0]
    return g2p

def edge_jaccard(gt_nodes, gt_edges, pred_nodes, pred_edges, max_um=MATCH_UM):
    g2p = match_nodes_per_t(gt_nodes, pred_nodes, max_um)
    p2g = {v: k for k, v in g2p.items()}
    pred_set, gt_set = set(map(tuple, pred_edges)), set(map(tuple, gt_edges))
    TP = FP = 0
    for a, b in pred_edges:
        if a in p2g and b in p2g:
            TP += (p2g[a], p2g[b]) in gt_set; FP += (p2g[a], p2g[b]) not in gt_set
    FN = sum(1 for u, v in gt_edges if not (u in g2p and v in g2p and (g2p[u], g2p[v]) in pred_set))
    d = TP + FP + FN
    return dict(jaccard=TP/d if d else 0.0, TP=TP, FP=FP, FN=FN,
                matched_gt=len(g2p), n_gt_nodes=len(gt_nodes), n_pred_nodes=len(pred_nodes))

In [ ]:
# ================= RUN: high-recall detect + physical NMS -> rule-based link -> score =================
model = UNet3D(base=BASE, strides=STRIDES).to(device)
wp = find_weights()
assert wp, 'no .pt found under /kaggle/input -- add your weights dataset (v1_UNet_best.pt) as an input.'
model.load_state_dict(extract_state(torch.load(wp, map_location=device))); model.eval()
print('loaded weights:', wp)
print(f'preset: HM_THR={HM_THR} min_dist={MIN_DIST} NMS_UM={NMS_UM} loose={LOOSE_UM} filter_min_len={FILTER_MIN_LEN}\n')

samples = build_sample_list(TRAIN_DIR)
val_s = samples[-N_VAL:]

TP = FP = FN = 0; t0 = time.time()
for zp, gp in val_s[:EVAL_N_VAL]:
    arr = open_image(zp)
    frames = detect_frames(model, arr)
    g = link_twopass(frames, tight_um=TIGHT_UM, loose_um=LOOSE_UM, vel_blend=VEL_BLEND)
    if CLOSE_GAPS:     g = close_gaps(frames, g, max_gap=MAX_GAP, gap_dist_um=GAP_DIST_UM)
    if PRUNE_ISOLATED: g = prune_isolated(g)
    if FILTER_MIN_LEN > 1: g = filter_short_tracks(g, FILTER_MIN_LEN)
    p_nodes, p_edges = graph_to_nodes_edges(g)
    gt_nodes, gt_edges = gt_nodes_edges(gp)
    r = edge_jaccard(gt_nodes, gt_edges, p_nodes, p_edges)
    TP += r['TP']; FP += r['FP']; FN += r['FN']
    print(f"  {os.path.basename(gp)[:-5]}: J={r['jaccard']:.3f} TP={r['TP']} FP={r['FP']} FN={r['FN']} "
          f"matched {r['matched_gt']}/{r['n_gt_nodes']} pred_nodes={r['n_pred_nodes']}")

micro = TP / (TP + FP + FN + 1e-9)
print(f"\nmicro edge-Jaccard: {micro:.4f}  (TP={TP} FP={FP} FN={FN})  [{time.time()-t0:.0f}s]")
print("compare -> v2 (HM_THR=0.3, no NMS, min_len=4): 0.859  |  v1 NN linking: 0.808")

**How to read it.** Compare to **v2 = 0.859** (same detector, same val samples):
- **`matched/n_gt` (detection recall) should rise** — that is the point of the high-recall preset.
  If edge-Jaccard rises with it, recall converted to real edges. If recall rises but J is flat/down,
  the extra detections are noise (or NMS merged real cells / linking got confused) -> retune.
- **Watch FP.** v2 had FP~0. If denser detection + looser linking pushes FP up, tighten `LOOSE_UM` back
  toward 8 or raise `NMS_UM`.

**Ablation order (change one knob, re-run the RUN cell):**
1. `NMS_UM` — 0 (off) vs 3.0 vs 4.0. Larger = fewer duplicates but risks merging close real cells.
2. `HM_THR` — 0.15 vs 0.2 vs 0.25. Lower = more recall but more noise + slower.
3. `LOOSE_UM` — 8 vs 10. 4. `FILTER_MIN_LEN` — 1 (off) vs 2 vs 4.

**If a preset beats 0.859:** port the same detection change (physical NMS + threshold) into `src/submit.ipynb`
and submit. Keep `EVAL_N_VAL` at 5 for a direct v2 comparison first, then raise to 20 to confirm.